## Vector Search Index Sync
File    : pipeline/05_sync_index.ipynb
Purpose : Triggers a sync of the Vector Search index after
          new news articles arrive in main.silver.news_for_search
Run     : Automatically via Databricks Workflow after Silver step


In [ ]:
# 0. Install package
%pip install databricks-ai-search --quiet
dbutils.library.restartPython()


In [ ]:
# 1. Update news_for_search source table with latest Silver data
# Unions news articles + company profiles (spec: embed both)
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from databricks.ai_search.client import VectorSearchClient
from datetime import datetime
import time

spark = SparkSession.builder.getOrCreate()

print(f"Starting VS sync: {datetime.now().isoformat()}")

# ── Part A: News articles ─────────────────────────────────────────────────────
print("\n--- Refreshing news_for_search (news + company profiles) ---")

news_for_search = (
    spark.table("main.silver.news_articles")
    .withColumn("search_text",
        F.concat_ws(" | ",
            F.col("ticker"),
            F.col("title"),
            F.coalesce(F.col("description"), F.lit(""))
        )
    )
    .select(
        "article_id", "ticker", "title", "description",
        "search_text", "publisher_name", "sentiment",
        "published_utc", "published_ts",
        "article_age_days", "has_sentiment", "processed_at"
    )
    .filter(F.col("article_id").isNotNull())
    .filter(F.col("search_text").isNotNull())
)

news_count = news_for_search.count()
print(f"News articles: {news_count} rows")

# ── Part B: Company profiles ──────────────────────────────────────────────────
companies_for_search = (
    spark.table("main.silver.companies")
    .filter(F.col("description").isNotNull())
    .withColumn("search_text",
        F.concat_ws(" | ",
            F.col("ticker"),
            F.col("name"),
            F.coalesce(F.col("description"), F.lit(""))
        )
    )
    .withColumn("article_id",
        F.concat(F.lit("company_"), F.col("ticker"))
    )
    .withColumn("title",          F.col("name"))
    .withColumn("publisher_name", F.lit("Company Profile"))
    .withColumn("sentiment",      F.lit("neutral"))
    .withColumn("published_utc",  F.lit(datetime.now().strftime("%Y-%m-%d")))
    .withColumn("published_ts",   F.to_timestamp(F.lit(datetime.now().strftime("%Y-%m-%d"))))
    .withColumn("article_age_days", F.lit(0))
    .withColumn("has_sentiment",  F.lit(False))
    .withColumn("processed_at",   F.col("processed_at"))
    .select(
        "article_id", "ticker", "title", "description",
        "search_text", "publisher_name", "sentiment",
        "published_utc", "published_ts",
        "article_age_days", "has_sentiment", "processed_at"
    )
    .filter(F.col("search_text").isNotNull())
)

company_count = companies_for_search.count()
print(f"Company profiles: {company_count} rows")

# ── Union + write ─────────────────────────────────────────────────────────────
combined = news_for_search.union(companies_for_search)

(combined
 .write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("main.silver.news_for_search"))

total = spark.table("main.silver.news_for_search").count()
print(f"Updated {total} rows in news_for_search ✓")
print(f"  - News articles    : {news_count}")
print(f"  - Company profiles : {company_count}")


In [ ]:
# 2. Trigger Vector Search index sync
print("\n--- Triggering Vector Search sync ---")

vsc = VectorSearchClient(disable_notice=True)

try:
    idx = vsc.get_index("stock-assistant-vs", "main.silver.news_for_search_index")
    idx.sync()
    print("Index sync triggered ✓")
    print("Waiting 60s for sync to complete...")
    time.sleep(60)

    desc  = idx.describe()
    state = desc.get("status", {}).get("detailed_state", "UNKNOWN")
    ready = desc.get("status", {}).get("ready", False)
    print(f"Index state: {state} | Ready: {ready}")
except Exception as e:
    print(f"Sync note: {e}")

print(f"\nVS sync complete: {datetime.now().isoformat()}")
